# DSP Speaker Embedding Experiment

Run this notebook from the project root after placing VCTK and MUSAN in the expected dataset layout. The notebook trains the same ECAPA-TDNN checkpoint under clean, noisy, and noisy-Wiener conditions, then evaluates speaker verification and clustering.

## Environment

Create the environment before starting Jupyter: `uv sync`. Then launch this notebook with `uv run jupyter lab main.ipynb`.

In [ ]:
from dataclasses import replace
import json
from pathlib import Path
from pprint import pprint

from src.config.settings import ExperimentConfig
from src.experiments.train import evaluate_checkpoint, prepare_manifests, train_condition

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open main.ipynb from the DSP project root.')

PROJECT_ROOT

## Signal Chain and Sampling Theory

VCTK `wav48` is the clean source dataset at 48 kHz, so its source Nyquist frequency is 24 kHz. The pretrained ECAPA encoder expects mono 16 kHz; the data loader resamples VCTK with anti-alias filtering before model input, so the model-observable band is $0$-$8\,\text{kHz}$:

$$
f_N = \frac{f_s}{2} = \frac{16{,}000}{2} = 8{,}000\,\text{Hz}.
$$

The often cited $20$-$20{,}000\,\text{Hz}$ range describes typical human hearing, not a requirement that this model must receive that whole range. Speech identity uses vocal fundamental frequency, harmonics, formants, and consonant detail across a broader spectrum; this experiment makes claims only about the $0$-$8\,\text{kHz}$ band delivered to ECAPA. A virtual microphone response is not simulated because VCTK provides no device-specific capture specification for this experiment.

## Dataset Setup

Download VCTK from the link in `dataset/README.md`, extract it as `dataset/VCTK-Corpus`, and place the MUSAN `noise` subset at `dataset/musan/noise`. VCTK and MUSAN have separate licenses; review and accept them before downloading.

Expected layout:

```text
dataset/
  VCTK-Corpus/wav48/p225/*.wav
  musan/noise/<source>/*
```

In [ ]:
# Change these values before creating manifests. Keep them identical for all conditions.
EXPERIMENT_OVERRIDES = {
    'seed': 42,
    'segment_seconds': 3.0,
    'batch_size': 16,
    'epochs': 10,
    'learning_rate': 1e-4,
    'snr_db': (5, 10, 15, 20),
    'wiener_window_size': 29,
}

CONDITIONS = ('clean_baseline', 'noisy', 'noisy_wiener')
configs = {
    condition: replace(ExperimentConfig(condition=condition), **EXPERIMENT_OVERRIDES)
    for condition in CONDITIONS
}

pprint({condition: config for condition, config in configs.items()})

In [ ]:
baseline_config = configs['clean_baseline']
required_paths = {
    'VCTK wav48': baseline_config.vctk_root,
    'MUSAN noise': baseline_config.musan_root / 'noise',
}
missing_paths = {name: path for name, path in required_paths.items() if not path.is_dir()}
if missing_paths:
    formatted = '\n'.join(f'- {name}: {path}' for name, path in missing_paths.items())
    raise FileNotFoundError(f'Missing required dataset directories:\n{formatted}')

print('Dataset preflight passed.')
pprint(required_paths)

## Prepare Speaker-Disjoint Manifests

Run this once after finalizing the seed, segment duration, and VCTK location. The manifests fix train, validation, and test speakers and deterministic crop positions for every subsequent condition.

In [ ]:
manifest_paths = prepare_manifests(baseline_config)
pprint(manifest_paths)

## Analyze Source, Composite Noise, and DSP Stages

The analysis runs before training on one deterministic held-out VCTK segment. It first compares the 48 kHz source segment with its 16 kHz ECAPA input, then constructs the exact composite noise used in phases 2 and 3:

- `environmental`: unfiltered MUSAN noise.
- `low_band`: separately selected MUSAN noise band-limited to $20$-$300\,\text{Hz}$.
- `high_band`: separately selected MUSAN noise band-limited to $3000$-$7500\,\text{Hz}$.

The reported SNR is for the total composite noise, not for each component. Only phase 3 continues from the same composite waveform through `high-pass 80 Hz -> low-pass 7500 Hz -> Wiener`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, sosfreqz, welch
import torch
import torchaudio

from src.audio import (
    CompositeMusanNoiseMixer,
    HighPassFilter,
    LowPassFilter,
    WienerDenoiser,
    mix_at_snr,
)
from src.data import VCTKWaveformDataset, load_manifest

analysis_record = load_manifest(manifest_paths['test'])[0]
source_waveform, source_sample_rate = torchaudio.load(analysis_record.audio_path)
source_waveform = source_waveform.mean(dim=0)
source_start = round(analysis_record.crop_start_seconds * source_sample_rate)
source_length = round(baseline_config.segment_seconds * source_sample_rate)
source_waveform = source_waveform[source_start : source_start + source_length]
source_waveform = torch.nn.functional.pad(
    source_waveform, (0, max(0, source_length - source_waveform.numel()))
)

analysis_dataset = VCTKWaveformDataset(
    [analysis_record],
    sample_rate=baseline_config.sample_rate,
    segment_seconds=baseline_config.segment_seconds,
)
clean_waveform = analysis_dataset[0]['waveform']
noise_root = baseline_config.musan_root / 'noise'
composite_mixer = CompositeMusanNoiseMixer.from_root(
    noise_root,
    seed=baseline_config.seed,
    snr_db=baseline_config.snr_db,
    low_noise_band_hz=baseline_config.low_noise_band_hz,
    high_noise_band_hz=baseline_config.high_noise_band_hz,
    filter_order=baseline_config.filter_order,
)
noise_sample = composite_mixer.build_noise(
    analysis_record,
    sample_rate=baseline_config.sample_rate,
    target_length=clean_waveform.numel(),
)
noisy_waveform = mix_at_snr(clean_waveform, noise_sample.composite, noise_sample.snr_db)
high_pass_waveform = HighPassFilter(
    baseline_config.high_pass_hz, baseline_config.filter_order
)(noisy_waveform, baseline_config.sample_rate, analysis_record)
low_pass_waveform = LowPassFilter(
    baseline_config.low_pass_hz, baseline_config.filter_order
)(high_pass_waveform, baseline_config.sample_rate, analysis_record)
wiener_waveform = WienerDenoiser(baseline_config.wiener_window_size)(
    low_pass_waveform, baseline_config.sample_rate, analysis_record
)

components = noise_sample.components
stages = {
    'Clean model input': clean_waveform,
    'Composite noisy': noisy_waveform,
    'After high-pass': high_pass_waveform,
    'After low-pass': low_pass_waveform,
    'After Wiener': wiener_waveform,
}


def rms_dbfs(waveform: torch.Tensor) -> float:
    return float(20 * torch.log10(waveform.square().mean().sqrt().clamp_min(1e-8)))


def residual_snr_db(reference: torch.Tensor, estimate: torch.Tensor) -> float:
    error = estimate - reference
    return float(
        10
        * torch.log10(
            reference.square().mean().clamp_min(1e-8)
            / error.square().mean().clamp_min(1e-8)
        )
    )


def band_energy_fraction(waveform: torch.Tensor, sample_rate: int, low_hz: float, high_hz: float) -> float:
    frequencies, power = welch(waveform.cpu().numpy(), fs=sample_rate, nperseg=512)
    in_band = (frequencies >= low_hz) & (frequencies < high_hz)
    return float(power[in_band].sum() / power.sum().clip(min=1e-12))


component_analysis = {
    name: {
        'source_path': str(noise_sample.source_paths[name]),
        'rms_dbfs': round(rms_dbfs(waveform), 2),
    }
    for name, waveform in components.items()
}
stage_analysis = {
    name: {
        'rms_dbfs': round(rms_dbfs(waveform), 2),
        'residual_snr_db_vs_clean': 'reference' if name == 'Clean model input' else round(
            residual_snr_db(clean_waveform, waveform), 2
        ),
    }
    for name, waveform in stages.items()
}
pprint(component_analysis)
pprint(stage_analysis)
print(f'Sample: {analysis_record.sample_id} ({analysis_record.speaker_id})')
print(f'Source rate: {source_sample_rate} Hz; source Nyquist: {source_sample_rate / 2:.0f} Hz')
print(f'Model rate: {baseline_config.sample_rate} Hz; model Nyquist: {baseline_config.nyquist_hz:.0f} Hz')
print(f'Total composite SNR: {noise_sample.snr_db} dB')

In [ ]:
target_noise_rms = clean_waveform.square().mean().sqrt() / (10 ** (noise_sample.snr_db / 20))
composite_scale = target_noise_rms / noise_sample.composite.square().mean().sqrt().clamp_min(1e-8)
pre_normalized_mix = clean_waveform + noise_sample.composite * composite_scale
peak_scale = pre_normalized_mix.abs().max().clamp_min(1.0)
components = {
    name: waveform * composite_scale / peak_scale
    for name, waveform in noise_sample.components.items()
}

print('Component contribution in the final composite mixture:')
pprint({name: {'rms_dbfs': round(rms_dbfs(waveform), 2)} for name, waveform in components.items()})
assert torch.allclose(
    clean_waveform / peak_scale + sum(components.values()),
    noisy_waveform,
)

In [ ]:
sample_rate = baseline_config.sample_rate
colors = {
    'environmental': '#7c3aed',
    'low_band': '#d97706',
    'high_band': '#dc2626',
    'Clean model input': '#2563eb',
    'Composite noisy': '#991b1b',
    'After high-pass': '#0f766e',
    'After low-pass': '#0369a1',
    'After Wiener': '#059669',
}

high_pass_sos = butter(
    baseline_config.filter_order,
    baseline_config.high_pass_hz / baseline_config.nyquist_hz,
    btype='highpass',
    output='sos',
)
low_pass_sos = butter(
    baseline_config.filter_order,
    baseline_config.low_pass_hz / baseline_config.nyquist_hz,
    btype='lowpass',
    output='sos',
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
source_frequency, source_power = welch(source_waveform.cpu().numpy(), fs=source_sample_rate, nperseg=2048)
model_frequency, model_power = welch(clean_waveform.cpu().numpy(), fs=sample_rate, nperseg=512)
axes[0, 0].semilogy(source_frequency, source_power, label='VCTK source segment (48 kHz)')
axes[0, 0].semilogy(model_frequency, model_power, label='ECAPA input (16 kHz)')
axes[0, 0].axvline(source_sample_rate / 2, color='#6b7280', linestyle=':', label='Source Nyquist')
axes[0, 0].axvline(baseline_config.nyquist_hz, color='#111827', linestyle='--', label='Model Nyquist')
axes[0, 0].set(title='Source bandwidth versus model bandwidth', xlabel='Frequency (Hz)', ylabel='PSD', xlim=(0, source_sample_rate / 2))
axes[0, 0].legend()

for name, waveform in components.items():
    frequency, power = welch(waveform.cpu().numpy(), fs=sample_rate, nperseg=512)
    axes[0, 1].semilogy(frequency, power, color=colors[name], label=name)
for cutoff in (*baseline_config.low_noise_band_hz, *baseline_config.high_noise_band_hz):
    axes[0, 1].axvline(cutoff, color='#374151', linestyle=':', alpha=0.7)
axes[0, 1].set(title='Composite-noise components at 16 kHz', xlabel='Frequency (Hz)', ylabel='PSD', xlim=(0, baseline_config.nyquist_hz))
axes[0, 1].legend()

for name, waveform in stages.items():
    frequency, power = welch(waveform.cpu().numpy(), fs=sample_rate, nperseg=512)
    axes[1, 0].semilogy(frequency, power, color=colors[name], label=name)
for cutoff in (baseline_config.high_pass_hz, baseline_config.low_pass_hz, baseline_config.nyquist_hz):
    axes[1, 0].axvline(cutoff, color='#374151', linestyle=':', alpha=0.7)
axes[1, 0].set(title='DSP stages in the frequency domain', xlabel='Frequency (Hz)', ylabel='PSD', xlim=(0, baseline_config.nyquist_hz))
axes[1, 0].legend(fontsize=8)

for sos, label, color in (
    (high_pass_sos, f'High-pass {baseline_config.high_pass_hz:.0f} Hz', '#0f766e'),
    (low_pass_sos, f'Low-pass {baseline_config.low_pass_hz:.0f} Hz', '#0369a1'),
):
    frequency, response = sosfreqz(sos, worN=2048, fs=sample_rate)
    axes[1, 1].plot(frequency, 20 * np.log10(np.maximum(np.abs(response), 1e-8)), label=label, color=color)
axes[1, 1].axvline(baseline_config.nyquist_hz, color='#111827', linestyle='--', label='Nyquist 8 kHz')
axes[1, 1].axvspan(baseline_config.low_pass_hz, baseline_config.nyquist_hz, color='#e5e7eb', alpha=0.6, label='Guard band')
axes[1, 1].set(title='Pre-Wiener filter responses', xlabel='Frequency (Hz)', ylabel='Gain (dB)', xlim=(0, baseline_config.nyquist_hz), ylim=(-80, 5))
axes[1, 1].legend(fontsize=8)
plt.show()

fig, axes = plt.subplots(len(stages), 1, figsize=(14, 13), constrained_layout=True)
for axis, (name, waveform) in zip(axes, stages.items(), strict=True):
    stft = torch.stft(
        waveform,
        n_fft=512,
        hop_length=160,
        window=torch.hann_window(512),
        return_complex=True,
    ).abs().square()
    image = axis.imshow(
        10 * torch.log10(stft.clamp_min(1e-10)).cpu().numpy(),
        origin='lower',
        aspect='auto',
        extent=[0, baseline_config.segment_seconds, 0, baseline_config.nyquist_hz],
        cmap='magma',
        vmin=-80,
        vmax=20,
    )
    axis.axhline(baseline_config.high_pass_hz, color='white', linestyle=':', linewidth=0.8)
    axis.axhline(baseline_config.low_pass_hz, color='white', linestyle=':', linewidth=0.8)
    axis.set(title=name, xlabel='Time (s)', ylabel='Frequency (Hz)')
fig.colorbar(image, ax=axes, label='Power (dB)')
plt.show()

bands = ((0, 80), (80, 300), (300, 3000), (3000, 7500), (7500, 8000))
band_energy = {
    name: {
        f'{low_hz}-{high_hz} Hz': round(
            band_energy_fraction(waveform, sample_rate, low_hz, high_hz), 4
        )
        for low_hz, high_hz in bands
    }
    for name, waveform in stages.items()
}
pprint(band_energy)

### Stage-by-Stage Interpretation

Use the measurements and plots to make a conditional statement for this selected segment:

- The raw composite must be evaluated using its **total** SNR; do not treat the component count as three independent SNR labels.
- High-pass should reduce energy below $80\,\text{Hz}$; low-pass should reduce energy from $7.5$ to $8\,\text{kHz}$.
- Wiener may improve residual SNR, but it can also reduce speaker-relevant speech detail. Claim a DSP benefit only when the signal-level evidence and later speaker-verification metrics agree.
- The clean reference uses a different, clean-test protocol and remains an upper reference rather than a head-to-head noisy score.

In [ ]:
stage_snr = {
    name: residual_snr_db(clean_waveform, waveform)
    for name, waveform in stages.items()
    if name != 'Clean model input'
}
print(f"Composite target SNR: {noise_sample.snr_db} dB")
for name, value in stage_snr.items():
    print(f'{name}: residual SNR versus paired clean input = {value:.2f} dB')

for name, waveform in stages.items():
    print(
        f"{name}: RMS={rms_dbfs(waveform):.2f} dBFS, "
        f"peak={waveform.abs().max().item():.4f}"
    )

wiener_recovery = stage_snr['After Wiener'] - stage_snr['Composite noisy']
if wiener_recovery > 0:
    print(f'Full DSP chain recovered {wiener_recovery:.2f} dB relative to composite noisy audio for this segment.')
else:
    print(
        f'Full DSP chain changed residual SNR by {wiener_recovery:.2f} dB for this segment. '
        'Inspect PSD/spectrogram and verification metrics before calling it an improvement.'
    )

## Fine-Tune Three Conditions

Each run starts from the same pretrained SpeechBrain ECAPA checkpoint and uses the same speaker-disjoint manifests.

1. `clean_baseline` receives clean 16 kHz VCTK input.
2. `noisy` receives the deterministic raw composite MUSAN mixture.
3. `noisy_wiener` receives the identical raw composite mixture, then `high-pass 80 Hz -> low-pass 7500 Hz -> Wiener` before ECAPA.

The composite source IDs and total SNR are deterministic for each sample ID, so phase 2 and phase 3 differ only after the DSP branch begins.

In [ ]:
train_results = {}
for condition, config in configs.items():
    print(f'\nTraining {condition} ...')
    train_results[condition] = train_condition(config)
    print(train_results[condition])

## Evaluate on Shared Composite-Noisy Test Audio

All checkpoints are evaluated on the same held-out composite noisy instances. `clean_baseline` and `noisy` receive raw composite noise; `noisy_wiener` receives that same composite after high-pass, low-pass, and Wiener processing. This is the direct robustness comparison.

In [ ]:
shared_noisy_results = {}
for condition, config in configs.items():
    checkpoint_path = train_results[condition].checkpoint_path
    shared_noisy_results[condition] = evaluate_checkpoint(config, checkpoint_path)
    print(f'\n{condition}')
    pprint(shared_noisy_results[condition])

## Clean Reference

Evaluate the clean baseline separately on clean held-out audio. This is a reference ceiling, not a direct head-to-head noisy test score.

In [ ]:
clean_reference_result = evaluate_checkpoint(
    configs['clean_baseline'],
    train_results['clean_baseline'].checkpoint_path,
    clean_reference=True,
)
pprint(clean_reference_result)

## Compare Results

Lower EER is better. Higher ROC-AUC, ARI, NMI, V-measure, and clustered coverage are better. HDBSCAN is corroborating evidence; use verification metrics for the primary conclusion.

In [ ]:
comparison = {
    condition: {
        **result.verification,
        **result.clustering,
    }
    for condition, result in shared_noisy_results.items()
}
comparison['clean_baseline_clean_reference'] = {
    **clean_reference_result.verification,
    **clean_reference_result.clustering,
}
print(json.dumps(comparison, indent=2, sort_keys=True))

## Model-Level Comparison Charts

These charts compare the three checkpoints under the shared noisy-test protocol. The clean-reference result is displayed separately because it uses clean test audio and must not be interpreted as a head-to-head noisy-test score.

In [ ]:
conditions = list(shared_noisy_results)
condition_labels = ['Clean baseline', 'Noisy', 'Noisy + Wiener']
condition_colors = ['#2563eb', '#dc2626', '#059669']
verification = [shared_noisy_results[condition].verification for condition in conditions]
clustering = [shared_noisy_results[condition].clustering for condition in conditions]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

axes[0].bar(condition_labels, [result['eer'] for result in verification], color=condition_colors)
axes[0].set(title='Verification error on shared noisy test', ylabel='EER (lower is better)', ylim=(0, 1))
axes[0].tick_params(axis='x', rotation=18)

axes[1].bar(condition_labels, [result['roc_auc'] for result in verification], color=condition_colors)
axes[1].set(title='Verification discrimination on shared noisy test', ylabel='ROC-AUC (higher is better)', ylim=(0, 1))
axes[1].tick_params(axis='x', rotation=18)

cluster_metrics = ('ari', 'nmi', 'v_measure')
positions = np.arange(len(condition_labels))
width = 0.24
for offset, metric in enumerate(cluster_metrics):
    axes[2].bar(
        positions + (offset - 1) * width,
        [result[metric] for result in clustering],
        width,
        label=metric.upper(),
    )
axes[2].set(
    title='HDBSCAN agreement with held-out speakers',
    ylabel='Score (higher is better)',
    xticks=positions,
    xticklabels=condition_labels,
    ylim=(-1, 1),
)
axes[2].tick_params(axis='x', rotation=18)
axes[2].legend()
plt.show()

fig, axis = plt.subplots(figsize=(7, 4), constrained_layout=True)
coverage = [result['clustered_coverage'] for result in clustering]
outliers = [result['outlier_rate'] for result in clustering]
axis.bar(condition_labels, coverage, label='Clustered coverage', color='#0f766e')
axis.bar(condition_labels, outliers, bottom=coverage, label='HDBSCAN outliers', color='#d97706')
axis.set(title='HDBSCAN assigned versus outlier samples', ylabel='Fraction', ylim=(0, 1))
axis.tick_params(axis='x', rotation=18)
axis.legend()
plt.show()

noisy_eer = shared_noisy_results['noisy'].verification['eer']
wiener_eer = shared_noisy_results['noisy_wiener'].verification['eer']
noisy_auc = shared_noisy_results['noisy'].verification['roc_auc']
wiener_auc = shared_noisy_results['noisy_wiener'].verification['roc_auc']
print(f'Noisy + Wiener versus noisy: EER delta={wiener_eer - noisy_eer:+.4f}; ROC-AUC delta={wiener_auc - noisy_auc:+.4f}.')
print(
    'A negative EER delta and positive ROC-AUC delta support improved noisy-speaker separation. '
    'Compare those values with the separate clean-reference report, not with a mixed protocol.'
)